# Ashok Gudivada CSET 01

## Starter Code (OpenCV + Python)

### Image Enhancement Pipeline

In [ ]:
%%capture
!pip install -q opencv-python numpy matplotlib pyflowchart
# This cell installs the required libraries for the image pipelines.
# suppressing the cells output

**Import for this section:**
  * `cv2` for image processing
  * `numpy` for numerical operations
  * `matplotlib.pyplot` for visualization
  * `Markdown` from `IPython.display` for formatted output in Jupyter notebooks

In [ ]:
import os
import cv2 as cv
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# Enable inline plotting for Jupyter
%matplotlib inline


In [ ]:
# Define the path to the images and create a list of image paths
img_path = "data/"
# Change the range of images as needed including images in the data folder
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
# Check if the images exist and are readable else remove the non-existing ones from the list
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Pipeline Steps:
# 1. White Balance / Color Correction
wb_imgs = []
wb_imgs_paths = []  # To store paths of saved white balanced images for next steps

# 2. Tone Lift (Gamma) + Contrast Enhancement (CLAHE)
tone_lift_imgs = []
tone_lift_imgs_paths = [] # To store paths of saved tone lifted images for next steps

# 3. Denoise (Edge-Preserving)
denoised_imgs = []
done_imgs_paths = [] # To store paths of saved denoised images for next steps

# 4. Sharpen (Unsharp Mask) with Clipping Control
sharpened_imgs = []
sharpened_imgs_paths = [] # To store paths of saved sharpened images for next steps

display(Markdown(f"**Found {len(img_path_array)} images:**"))
print("\n".join(img_path_array))
display(Markdown("**Image Sourced from https://pixabay.com/photos/**"))

# Step 1: White Balance / Color Correction

In [44]:
def imread_bgr(path):
    """Read an image from a file path in BGR format."""
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def exposure_map(img, low, high):
    """Compute underexposed and overexposed pixel maps based on thresholds."""
    low_thresh  = low
    high_thresh = high
    under = img < low_thresh
    over  = img > high_thresh
    return (under, over)

def bgr2rgb(img):
    """Convert BGR image to RGB format."""
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

def to_uint8(x):
    """Clip values to [0, 255] and convert to uint8."""
    return np.clip(x, 0, 255).astype(np.uint8)

def gamma_u8(u8, gamma):
    """Gamma correction for uint8 images."""
    x = u8.astype(np.float32) / 255.0
    y = np.power(x, gamma)
    return to_uint8(255.0 * y)

def clahe_u8(u8, clipLimit=2.0, tileGridSize=(8,8)):
    """Contrast Limited Adaptive Histogram Equalization (CLAHE) for local contrast enhancement."""
    op = cv.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    return op.apply(u8)

def contrast_stretch_percentile(u8, p_low=1, p_high=99):
    """Contrast stretching based on percentile thresholds."""
    lo, hi = np.percentile(u8, [p_low, p_high])
    if hi <= lo + 1e-9:
        return u8.copy()
    out = (u8.astype(np.float32) - lo) * (255.0 / (hi - lo))
    return to_uint8(out)

def unsharp_mask_u8(u8, sigma=1.0, alpha=1.0):
    """Unsharp masking for image sharpening."""
    blur = cv.GaussianBlur(u8, (0,0), sigmaX=sigma, sigmaY=sigma)
    detail = cv.subtract(u8, blur)
    sharp = cv.addWeighted(u8, 1.0, detail, alpha, 0.0)
    return sharp

def var_laplacian(u8):
    """Variance of the Laplacian as a measure of local contrast."""
    lap = cv.Laplacian(u8, cv.CV_64F, ksize=3)
    return float(lap.var())

def global_contrast(u8):
    """Standard deviation of pixel intensities as a measure of global contrast."""
    return float(u8.std())

def shadow_fraction(u8, thr=20):
    """Fraction of pixels with intensity below a threshold."""
    return float(np.mean(u8 < thr))

def show_rgb(img_bgr, title=""):
    """Utility function to display a BGR image as RGB."""
    plt.figure(figsize=(6,4))
    plt.imshow(bgr2rgb(img_bgr))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def show_gray(u8, title=""):
    """Utility function to display a grayscale image."""
    plt.figure(figsize=(6,4))
    plt.imshow(u8, cmap="gray", vmin=0, vmax=255)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

def print_images(img_array: list):
    """Utility function to read and display images from a list of
    paths or image arrays."""
    ## (a) Load the color image. If BGR, convert to RGB for display.
    for i, img in enumerate(img_array):
        img = imread_bgr(img) if isinstance(img, str) else img
        show_rgb(img, title=f"Image {i+1}")


def save_images(img_array, output_path, prefix):
    """Save images to files with a specified prefix in the filename.
    And return the list of saved file paths for reference.

    Args:
        img_array: List of images (BGR format)
        output_path: Directory to save images
        prefix: Filename prefix (e.g., 'wb' for white balanced)
    """
    print(f"Saving {len(img_array)} images to {output_path}")
    if not os.path.exists(output_path):
        print(f"Directory {output_path} does not exist, creating it")
        os.makedirs(output_path)
    else:
        print(f"Directory {output_path} already exists")
        # Optionally, clear existing files in the directory
        [os.remove(os.path.join(output_path, f)) for f in os.listdir(output_path)]
    for i, img in enumerate(img_array):
        filename = f"{output_path}{prefix}_{i+1}.jpg"
        cv.imwrite(filename, img)
        print(f"Saved: {filename}")
    # Return list of saved file paths for reference
    return [f"{output_path}{prefix}_{i+1}.jpg" for i in range(len(img_array))]

def display_processing_comparison(img_bgr, process_fn, process_name, img_index=None):
    """Display original and processed images side by side.

    Args:
        img_bgr: Input image in BGR format
        process_fn: Processing function to apply (e.g., white_balance)
        process_name: Name of the processing technique (e.g., "White Balanced")
    """
    processed_img = process_fn(img_bgr)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(bgr2rgb(img_bgr))
    title = f"Original Image {img_index}" if img_index else "Original Image"
    plt.title(title)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(bgr2rgb(processed_img))
    title = f"{process_name} Image {img_index}" if img_index else f"{process_name} Image"
    plt.title(title)
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    # Return the processed image for further steps if needed
    return processed_img

##### Image Processing Pipeline Functions #####

## White Balance / Color Correction ##
def white_balance(img_bgr):
    """Simple white balance using gray world assumption"""
    result = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB).astype(np.float32)
    avg_a = np.mean(result[:, :, 1])
    avg_b = np.mean(result[:, :, 2])
    result[:, :, 1] -= ((avg_a - 128) * (result[:, :, 0] / 255.0) * 1.2)
    result[:, :, 2] -= ((avg_b - 128) * (result[:, :, 0] / 255.0) * 1.2)
    result = np.clip(result, 0, 255).astype(np.uint8)
    return cv.cvtColor(result, cv.COLOR_LAB2BGR)

## Tone Lift (Gamma) + Contrast Enhancement (CLAHE) ##
def apply_gamma_clahe(img_bgr, gamma=0.6, clip_limit=3.0, tile_size=(8, 8)):
    """Apply gamma correction and CLAHE to an image."""
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]
    l_gamma = gamma_u8(l_channel, gamma)
    l_clahe = clahe_u8(l_gamma, clipLimit=clip_limit, tileGridSize=tile_size)
    img_lab[:, :, 0] = l_clahe
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)

### Denoise (Edge-Preserving) ##
def edge_preserving_denoise(img_bgr, d=9, sigmaColor=75, sigmaSpace=75):
    """Apply bilateral filter for edge-preserving denoising."""
    return cv.bilateralFilter(img_bgr, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)

### Sharpen (Unsharp Mask) with Clipping Control ##
def apply_unsharp_mask(img_bgr, sigma=1.5, alpha=1.2):
    """Apply unsharp mask sharpening to the L channel of an image.

    Args:
        img_bgr: Input image in BGR format
        sigma: Gaussian blur sigma (controls scale of sharpening)
        alpha: Sharpening strength (higher = more sharpening)

    Returns:
        Sharpened image in BGR format
    """
    # Convert to LAB, sharpen L channel, convert back
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]

    # Apply unsharp mask
    l_sharp = unsharp_mask_u8(l_channel, sigma=sigma, alpha=alpha)

    # Clip to prevent over-sharpening artifacts
    l_sharp = np.clip(l_sharp, 0, 255).astype(np.uint8)
    img_lab[:, :, 0] = l_sharp
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)

### Final Mild Contrast / Saturation Adjustment ##
def apply_final_adjustment(img_bgr, saturation_factor=1.15, p_low=2, p_high=98):
    """Apply mild contrast and saturation adjustment.

    Args:
        img_bgr: Input image in BGR format
        saturation_factor: Factor to multiply saturation (>1 increases, <1 decreases)
        p_low: Low percentile for contrast stretch
        p_high: High percentile for contrast stretch

    Returns:
        Final adjusted image in BGR format
    """
    # Convert to HSV for saturation and value adjustments
    img_hsv = cv.cvtColor(img_bgr, cv.COLOR_BGR2HSV).astype(np.float32)

    # Increase saturation slightly
    img_hsv[:, :, 1] = np.clip(img_hsv[:, :, 1] * saturation_factor, 0, 255)

    # Mild contrast on value channel
    v_channel = img_hsv[:, :, 2]
    v_stretched = contrast_stretch_percentile(v_channel.astype(np.uint8), p_low=p_low, p_high=p_high)
    img_hsv[:, :, 2] = v_stretched

    # Convert back to BGR
    img_final = cv.cvtColor(np.clip(img_hsv, 0, 255).astype(np.uint8), cv.COLOR_HSV2BGR)
    return img_final

### Final Mild Contrast / Saturation Adjustment with Percentile Clipping ###
def apply_final_adjustment(img_bgr, saturation_factor=1.15, p_low=2, p_high=98):
    """Apply mild contrast and saturation adjustment."""
    img_hsv = cv.cvtColor(img_bgr, cv.COLOR_BGR2HSV).astype(np.float32)
    img_hsv[:, :, 1] = np.clip(img_hsv[:, :, 1] * saturation_factor, 0, 255)
    v_channel = img_hsv[:, :, 2]
    v_stretched = contrast_stretch_percentile(v_channel.astype(np.uint8), p_low=p_low, p_high=p_high)
    img_hsv[:, :, 2] = v_stretched
    return cv.cvtColor(np.clip(img_hsv, 0, 255).astype(np.uint8), cv.COLOR_HSV2BGR)

### Compute a grayscale version of the image (luminance proxy).
def gray_scale(img_bgr):
# Convert to grayscale
    import cv2
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray

In [ ]:


wb_imgs = []  # Re-initialize as empty list
for i, img_path in enumerate(img_path_array):
    img = imread_bgr(img_path)
    wb_imgs.append(display_processing_comparison(img, white_balance, "White Balanced", i+1))

wb_imgs_paths = []
# Save the white balanced images to files for later use
wb_imgs_paths = save_images(wb_imgs, "data/white_balanced/", "wb")





In [ ]:
print(wb_imgs_paths)

# Step 2: Tone Lift (Gamma) + Contrast Enhancement (CLAHE)


In [ ]:
# Apply gamma correction and CLAHE to all white-balanced images
gamma_value = 0.6  # values < 1 brighten the image
tone_lift_imgs = []  # Re-initialize as empty list
tone_lift_imgs_paths = []


for i, img_path in enumerate(wb_imgs_paths):
    img = imread_bgr(img_path)
    tone_lift_imgs.append(display_processing_comparison(img, apply_gamma_clahe, "Gamma + CLAHE", i+1))

# Save the Gamma + CLAHE images to files for later use
tone_lift_imgs_paths = save_images(tone_lift_imgs, "data/tone_lift/", "gc")




# Step 3: Denoise (Edge-Preserving)

In [ ]:
# Use bilateral filter for edge-preserving denoising
denoised_imgs = []  # Initialize the list to store denoised images
done_imgs_paths = [] # To store paths of saved denoised images for later use
for i, img_path in enumerate(tone_lift_imgs_paths):
    img = imread_bgr(img_path)
    denoised_imgs.append(display_processing_comparison(img, edge_preserving_denoise, "Denoised", i+1))

# Save the denoised images to files for later use
done_imgs_paths = save_images(denoised_imgs, "data/denoised/", "ep")

# Step 4: Sharpen (Unsharp Mask) with Clipping Control

In [ ]:
# Apply sharpening to all denoised images
sharpened_imgs = []  # Initialize the list to store sharpened images
sharpened_imgs_paths = []  # To store paths of saved sharpened images for later use

for i, img_path in enumerate(done_imgs_paths):
    img = imread_bgr(img_path)
    sharpened_imgs.append(display_processing_comparison(img, apply_unsharp_mask, "Sharpened", i+1))

# Save the sharpened images to files for later use
sharpened_imgs_paths = save_images(sharpened_imgs, "data/sharpened/", "sh")


# Step 5: Final Mild Contrast / Saturation Adjustment

In [ ]:
# Apply final adjustment to all sharpened images
final_imgs = []  # Initialize the list to store final images
final_imgs_paths = []  # To store paths of saved final images

for i, img_path in enumerate(sharpened_imgs_paths):
    img = imread_bgr(img_path)
    final_imgs.append(display_processing_comparison(img, apply_final_adjustment, "Final Enhanced", i+1))

# Save the final enhanced images to files
final_imgs_paths = save_images(final_imgs, "data/contrast_saturation/", "cs")


## Image Enhancement Pipeline Flowchart


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Create directed graph for the pipeline
G = nx.DiGraph()

# Define nodes with labels
nodes = {
    'input': 'INPUT\nNighttime Image',
    'step1': 'Step 1\nWhite Balance\n(LAB Color Space)',
    'step2': 'Step 2\nGamma + CLAHE\n(Tone Lift)',
    'step3': 'Step 3\nDenoise\n(Bilateral Filter)',
    'step4': 'Step 4\nSharpen\n(Unsharp Mask)',
    'step5': 'Step 5\nContrast/Saturation\n(HSV Adjustment)',
    'output': 'OUTPUT\nEnhanced Image'
}

# Add nodes
for node_id, label in nodes.items():
    G.add_node(node_id, label=label)

# Add edges (connections)
edges = [
    ('input', 'step1'),
    ('step1', 'step2'),
    ('step2', 'step3'),
    ('step3', 'step4'),
    ('step4', 'step5'),
    ('step5', 'output')
]
G.add_edges_from(edges)

# Create figure
fig, ax = plt.subplots(figsize=(14, 10))

# Position nodes vertically (top to bottom flow)
pos = {
    'input': (0.5, 1.0),
    'step1': (0.5, 0.83),
    'step2': (0.5, 0.66),
    'step3': (0.5, 0.49),
    'step4': (0.5, 0.32),
    'step5': (0.5, 0.15),
    'output': (0.5, 0.0)
}

# Define colors for different node types
node_colors = {
    'input': '#90EE90',   # Light green
    'step1': '#87CEEB',   # Sky blue
    'step2': '#87CEEB',   # Sky blue
    'step3': '#87CEEB',   # Sky blue
    'step4': '#87CEEB',   # Sky blue
    'step5': '#87CEEB',   # Sky blue
    'output': '#FFB6C1'   # Light pink
}

# Draw nodes as rectangles with text
for node_id, (x, y) in pos.items():
    color = node_colors[node_id]
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color, edgecolor='black', linewidth=2)
    ax.text(x, y, nodes[node_id], ha='center', va='center', fontsize=10,
            fontweight='bold', bbox=bbox, transform=ax.transAxes)

# Draw edges (arrows)
for edge in edges:
    start_pos = pos[edge[0]]
    end_pos = pos[edge[1]]
    ax.annotate('', xy=(end_pos[0], end_pos[1] + 0.04), xytext=(start_pos[0], start_pos[1] - 0.04),
                arrowprops=dict(arrowstyle='->', color='#333333', lw=2),
                xycoords='axes fraction', textcoords='axes fraction')

# Add title and legend
ax.set_title('Night Image Enhancement Pipeline', fontsize=16, fontweight='bold', pad=20)

# Create legend
legend_elements = [
    mpatches.Patch(facecolor='#90EE90', edgecolor='black', label='Input'),
    mpatches.Patch(facecolor='#87CEEB', edgecolor='black', label='Processing Steps'),
    mpatches.Patch(facecolor='#FFB6C1', edgecolor='black', label='Output')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

# Add color space workflow annotation
ax.text(0.02, 0.5, 'Color Space Flow:\nBGR → LAB → HSV → BGR',
        transform=ax.transAxes, fontsize=9, verticalalignment='center',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

ax.axis('off')
plt.tight_layout()
plt.savefig('data/pipeline_flowchart.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("Flowchart saved to: data/pipeline_flowchart.png")


# Task 1: Baseline + Diagnostics

### (a) Load the color image. If BGR, convert to RGB for display.

In [ ]:
print_images(img_path_array)

#### (b) Compute a grayscale version of the image (luminance proxy).

In [ ]:
gray_imgs = []  # Initialize list to store grayscale images
gray_imgs_paths = []  # To store paths of saved grayscale images

print(f"Processing {len(final_imgs_paths)} images for grayscale conversion...")

for i, img_path in enumerate(final_imgs_paths):
    try:
        # Load image
        img = imread_bgr(img_path)
        if img is None:
            print(f"Error reading {img_path}")
            continue
            
        # Convert to grayscale
        # Use cv2 directly (assuming it is imported as cv or cv2)
        # If cv is used elsewhere, we use cv. If cv2, we use cv2. Trying cv2 first then cv.
        try:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        except NameError:
            import cv2
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Append valid grayscale image
        gray_imgs.append(gray)
        
        # Display
        plt.figure(figsize=(10, 5))
        
        # Original
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f'Original {i+1}')
        plt.axis('off')
        
        # Grayscale
        plt.subplot(1, 2, 2)
        plt.imshow(gray, cmap='gray')
        plt.title(f'Grayscale {i+1}')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Failed to process image {i+1}: {str(e)}")

# Save the grayscale images to files
if gray_imgs:
    gray_imgs_paths = save_images(gray_imgs, "data/grayscale/", "gray")
    print(f"Saved {len(gray_imgs_paths)} grayscale images.")
else:
    print("No valid grayscale images to save.")


#### (c) Save a baseline montage: original RGB, grayscale, and an “exposure map” e.g. show pixels below some minimum and above some maximum,, e.g. 20 and 235, respectively). An exposure map is simply a diagnostic visualization that flags under-exposed (too dark) and over-exposed (too bright / saturated)pixels.

In [ ]:
# Create baseline montage: Original RGB, Grayscale, and Exposure Map

def create_exposure_map_visual(gray_img, low_thresh=20, high_thresh=235):
    """Create a color-coded exposure map visualization.

    Args:
        gray_img: Grayscale image (uint8)
        low_thresh: Threshold below which pixels are considered underexposed
        high_thresh: Threshold above which pixels are considered overexposed

    Returns:
        Color image with underexposed (blue), normal (green), overexposed (red) regions
    """
    exposure_vis = np.zeros((gray_img.shape[0], gray_img.shape[1], 3), dtype=np.uint8)

    # Under-exposed pixels (blue)
    under = gray_img < low_thresh
    exposure_vis[under] = [0, 0, 255]

    # Over-exposed pixels (red)
    over = gray_img > high_thresh
    exposure_vis[over] = [255, 0, 0]

    # Normal exposure (green)
    normal = ~under & ~over
    exposure_vis[normal] = [0, 255, 0]

    return exposure_vis

def save_baseline_montage(img_paths, output_dir="data/baseline_montage/", low_thresh=20, high_thresh=235):
    """Create and save baseline montage for each image."""
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    montage_paths = []

    for i, img_path in enumerate(img_paths):
        # Load original image
        img_bgr = imread_bgr(img_path)
        img_rgb = bgr2rgb(img_bgr)

        # Convert to grayscale
        gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

        # Create exposure map visualization
        exposure_vis = create_exposure_map_visual(gray, low_thresh, high_thresh)

        # Calculate exposure statistics
        under_pct = (np.sum(gray < low_thresh) / gray.size) * 100
        over_pct = (np.sum(gray > high_thresh) / gray.size) * 100

        # Create montage figure
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original RGB
        axes[0].imshow(img_rgb)
        axes[0].set_title(f'Original RGB (Image {i+1})', fontsize=12, fontweight='bold')
        axes[0].axis('off')

        # Grayscale
        axes[1].imshow(gray, cmap='gray')
        axes[1].set_title(f'Grayscale (Image {i+1})', fontsize=12, fontweight='bold')
        axes[1].axis('off')

        # Exposure Map
        axes[2].imshow(exposure_vis)
        axes[2].set_title(f'Exposure Map\nUnder: {under_pct:.1f}% | Over: {over_pct:.1f}%',
                         fontsize=12, fontweight='bold')
        axes[2].axis('off')

        # Add legend for exposure map
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='blue', label=f'Underexposed (<{low_thresh})'),
            Patch(facecolor='green', label=f'Normal ({low_thresh}-{high_thresh})'),
            Patch(facecolor='red', label=f'Overexposed (>{high_thresh})')
        ]
        axes[2].legend(handles=legend_elements, loc='lower right', fontsize=8)

        plt.suptitle(f'Baseline Diagnostics - Image {i+1}', fontsize=14, fontweight='bold')
        plt.tight_layout()

        # Save montage
        montage_path = f"{output_dir}baseline_montage_{i+1}.png"
        plt.savefig(montage_path, dpi=150, bbox_inches='tight', facecolor='white')
        montage_paths.append(montage_path)

        plt.show()

        print(f"Image {i+1}: Underexposed: {under_pct:.2f}%, Overexposed: {over_pct:.2f}%")

    print(f"\nSaved {len(montage_paths)} baseline montages to {output_dir}")
    return montage_paths

# Generate baseline montages for all original images
# Execute baseline montage generation
display(Markdown("---"))
display(Markdown("## Baseline Montage Generation"))
display(Markdown(f"**Found {len(img_path_array)} images to process**"))
display(Markdown("""
**Exposure Map Color Coding:**
- **Blue:** Underexposed pixels (< 20)
- **Green:** Normal exposure (20-235)
- **Red:** Overexposed pixels (> 235)
"""))
display(Markdown("---"))

baseline_montage_paths = save_baseline_montage(img_path_array, low_thresh=20, high_thresh=235)

display(Markdown("---"))
display(Markdown(f"**Generated {len(baseline_montage_paths)} baseline montages**"))
display(Markdown(f"**Saved to:** `data/baseline_montage/`"))

# Task 2: Point Processing for Exposure and Contrast (30%)
Work in a way that avoids ruining color:


#### Convert the image to Lab (or HSV) and apply tone/contrast changes to the L (or V) channel only.

In [ ]:
def apply_tone_contrast_lab(img_bgr, gamma=0.7, clip_limit=2.5, tile_size=(8, 8)):
    """
    Apply tone and contrast adjustments to L channel only (LAB color space).

    Parameters
    img_bgr : numpy.ndarray
        Input image in BGR format
    gamma : float
        Gamma correction value (< 1 brightens, > 1 darkens)
    clip_limit : float
        CLAHE clip limit for contrast enhancement
    tile_size : tuple
        CLAHE tile grid size

    Returns
    numpy.ndarray
        Processed image in BGR format
    """
    # Convert BGR to LAB
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)

    # Extract L channel (luminance)
    l_channel = img_lab[:, :, 0]

    # Apply gamma correction to L channel
    l_gamma = gamma_u8(l_channel, gamma)

    # Apply CLAHE to L channel for local contrast enhancement
    l_clahe = clahe_u8(l_gamma, clipLimit=clip_limit, tileGridSize=tile_size)

    # Replace L channel with processed version
    img_lab[:, :, 0] = l_clahe

    # Convert back to BGR
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)


def apply_tone_contrast_hsv(img_bgr, gamma=0.7, clip_limit=2.5, tile_size=(8, 8)):
    """
    Apply tone and contrast adjustments to V channel only (HSV color space).

    Parameters
    ----------
    img_bgr : numpy.ndarray
        Input image in BGR format
    gamma : float
        Gamma correction value (< 1 brightens, > 1 darkens)
    clip_limit : float
        CLAHE clip limit for contrast enhancement
    tile_size : tuple
        CLAHE tile grid size

    Returns
    -------
    numpy.ndarray
        Processed image in BGR format
    """
    # Convert BGR to HSV
    img_hsv = cv.cvtColor(img_bgr, cv.COLOR_BGR2HSV)

    # Extract V channel (value/brightness)
    v_channel = img_hsv[:, :, 2]

    # Apply gamma correction to V channel
    v_gamma = gamma_u8(v_channel, gamma)

    # Apply CLAHE to V channel for local contrast enhancement
    v_clahe = clahe_u8(v_gamma, clipLimit=clip_limit, tileGridSize=tile_size)

    # Replace V channel with processed version
    img_hsv[:, :, 2] = v_clahe

    # Convert back to BGR
    return cv.cvtColor(img_hsv, cv.COLOR_HSV2BGR)


def compare_lab_vs_hsv(img_paths, gamma=0.7, clip_limit=2.5):
    """Compare LAB L-channel vs HSV V-channel processing."""

    display(Markdown("## 🎨 Tone/Contrast Adjustment: LAB vs HSV"))
    display(Markdown(f"**Parameters:** Gamma = `{gamma}` | CLAHE Clip Limit = `{clip_limit}`"))

    for i, img_path in enumerate(img_paths):
        img_bgr = imread_bgr(img_path)

        # Process using both methods
        img_lab_processed = apply_tone_contrast_lab(img_bgr, gamma, clip_limit)
        img_hsv_processed = apply_tone_contrast_hsv(img_bgr, gamma, clip_limit)

        # Display comparison
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(bgr2rgb(img_bgr))
        axes[0].set_title(f'Original (Image {i+1})', fontweight='bold')
        axes[0].axis('off')

        axes[1].imshow(bgr2rgb(img_lab_processed))
        axes[1].set_title(f'LAB (L-channel)', fontweight='bold')
        axes[1].axis('off')

        axes[2].imshow(bgr2rgb(img_hsv_processed))
        axes[2].set_title(f'HSV (V-channel)', fontweight='bold')
        axes[2].axis('off')

        plt.suptitle(f'Tone/Contrast Comparison - Image {i+1}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        display(Markdown(f"**Image {i+1}:** LAB preserves perceptual uniformity, HSV may shift colors slightly"))

# Execute comparison
display(Markdown("---"))
compare_lab_vs_hsv(img_path_array, gamma=0.7, clip_limit=2.5)
display(Markdown("---"))
display(Markdown("✅ **Recommendation:** Use LAB for perceptually uniform adjustments"))

#### Implement and compare:
1. Gamma lift on luminance: L′ = 255 (L/255)γ with γ ∈ {0.4, 0.6, 0.8}.
2. CLAHE on luminance with at least two parameter settings.
3. Percentile contrast stretch on luminance (e.g. 1%–99%).

In [ ]:
# ============================================================================
# TASK 2: TONE & CONTRAST ENHANCEMENT
# ============================================================================
# Implement and compare three enhancement methods on the L channel:
#   1. Gamma lift: L' = 255 * (L/255)^γ with γ ∈ {0.4, 0.6, 0.8}
#   2. CLAHE with at least two parameter settings
#   3. Percentile contrast stretch (1%-99%)
# ============================================================================

from IPython.display import Markdown, display


def apply_gamma_only(img_bgr, gamma):
    """Apply gamma correction to L channel only: L' = 255 * (L/255)^γ"""
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]
    l_gamma = gamma_u8(l_channel, gamma)
    img_lab[:, :, 0] = l_gamma
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)

def apply_clahe_only(img_bgr, clip_limit, tile_size=(8, 8)):
    """Apply CLAHE to L channel only."""
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]
    l_clahe = clahe_u8(l_channel, clipLimit=clip_limit, tileGridSize=tile_size)
    img_lab[:, :, 0] = l_clahe
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)

def apply_percentile_stretch_lab(img_bgr, p_low=1, p_high=99):
    """Apply percentile contrast stretch to L channel only."""
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]
    l_stretched = contrast_stretch_percentile(l_channel, p_low=p_low, p_high=p_high)
    img_lab[:, :, 0] = l_stretched
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)

# ============================================================================
# COMPARISON MONTAGE FUNCTION
# ============================================================================
def create_enhancement_comparison_montage(img_paths, output_dir="data/enhancement_comparison/"):
    """Create montage comparing 4+ enhancement methods."""

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    display(Markdown("# 🔬 Task 2: Tone & Contrast Enhancement Comparison"))

    montage_paths = []

    for i, img_path in enumerate(img_paths):
        img_bgr = imread_bgr(img_path)

        # =====================================================================
        # (1) GAMMA LIFT COMPARISON: γ ∈ {0.4, 0.6, 0.8}
        # =====================================================================
        display(Markdown(f"### 📊 Image {i+1}: (1) Gamma Lift Comparison"))
        display(Markdown("**Formula:** $L' = 255 \\times (L/255)^\\gamma$"))

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))

        axes[0].imshow(bgr2rgb(img_bgr))
        axes[0].set_title('Original', fontweight='bold')
        axes[0].axis('off')

        for idx, gamma in enumerate([0.4, 0.6, 0.8]):
            img_gamma = apply_gamma_only(img_bgr, gamma)
            axes[idx + 1].imshow(bgr2rgb(img_gamma))
            axes[idx + 1].set_title(f'γ = {gamma}', fontweight='bold')
            axes[idx + 1].axis('off')

        plt.suptitle(f'Gamma Lift on Luminance - Image {i+1}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{output_dir}gamma_comparison_{i+1}.png', dpi=150, bbox_inches='tight')
        plt.show()

        # =====================================================================
        # (2) CLAHE COMPARISON: Two+ parameter settings
        # =====================================================================
        display(Markdown(f"### 📊 Image {i+1}: (2) CLAHE Comparison"))

        clahe_params = [
            {'clip': 1.5, 'tile': (8, 8)},
            {'clip': 2.5, 'tile': (8, 8)},
            {'clip': 2.0, 'tile': (4, 4)},
            {'clip': 2.0, 'tile': (16, 16)}
        ]

        fig, axes = plt.subplots(1, 5, figsize=(20, 4))

        axes[0].imshow(bgr2rgb(img_bgr))
        axes[0].set_title('Original', fontweight='bold')
        axes[0].axis('off')

        for idx, params in enumerate(clahe_params):
            img_clahe = apply_clahe_only(img_bgr, params['clip'], params['tile'])
            axes[idx + 1].imshow(bgr2rgb(img_clahe))
            axes[idx + 1].set_title(f"Clip={params['clip']}\nTile={params['tile'][0]}×{params['tile'][1]}",
                                    fontweight='bold', fontsize=9)
            axes[idx + 1].axis('off')

        plt.suptitle(f'CLAHE on Luminance - Image {i+1}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{output_dir}clahe_comparison_{i+1}.png', dpi=150, bbox_inches='tight')
        plt.show()

        # =====================================================================
        # (3) PERCENTILE CONTRAST STRETCH
        # =====================================================================
        display(Markdown(f"### 📊 Image {i+1}: (3) Percentile Contrast Stretch"))

        percentile_params = [(1, 99), (2, 98), (5, 95)]

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))

        axes[0].imshow(bgr2rgb(img_bgr))
        axes[0].set_title('Original', fontweight='bold')
        axes[0].axis('off')

        for idx, (p_low, p_high) in enumerate(percentile_params):
            img_stretch = apply_percentile_stretch_lab(img_bgr, p_low, p_high)
            axes[idx + 1].imshow(bgr2rgb(img_stretch))
            axes[idx + 1].set_title(f'{p_low}% - {p_high}%', fontweight='bold')
            axes[idx + 1].axis('off')

        plt.suptitle(f'Percentile Contrast Stretch on Luminance - Image {i+1}', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{output_dir}percentile_comparison_{i+1}.png', dpi=150, bbox_inches='tight')
        plt.show()

        # =====================================================================
        # COMBINED MONTAGE: 4+ Best Results
        # =====================================================================
        display(Markdown(f"### 🏆 Image {i+1}: Combined Comparison Montage"))

        results = {
            'Original': img_bgr,
            'Gamma (γ=0.6)': apply_gamma_only(img_bgr, 0.6),
            'CLAHE (Clip=2.5)': apply_clahe_only(img_bgr, 2.5, (8, 8)),
            'Percentile (1%-99%)': apply_percentile_stretch_lab(img_bgr, 1, 99),
            'Gamma + CLAHE': apply_clahe_only(apply_gamma_only(img_bgr, 0.6), 2.0, (8, 8)),
            'All Combined': apply_percentile_stretch_lab(
                apply_clahe_only(apply_gamma_only(img_bgr, 0.6), 2.0, (8, 8)), 2, 98)
        }

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()

        for idx, (name, img) in enumerate(results.items()):
            axes[idx].imshow(bgr2rgb(img))
            axes[idx].set_title(name, fontsize=12, fontweight='bold')
            axes[idx].axis('off')

        plt.suptitle(f'Enhancement Comparison Montage - Image {i+1}', fontsize=16, fontweight='bold')
        plt.tight_layout()

        montage_path = f'{output_dir}combined_montage_{i+1}.png'
        plt.savefig(montage_path, dpi=150, bbox_inches='tight', facecolor='white')
        montage_paths.append(montage_path)
        plt.show()

        display(Markdown("---"))

    return montage_paths

# ============================================================================
# EXECUTE COMPARISON
# ============================================================================
display(Markdown("---"))
display(Markdown(f"**Processing {len(img_path_array)} images...**"))

enhancement_montage_paths = create_enhancement_comparison_montage(img_path_array)

display(Markdown("## 📝 Summary"))
display(Markdown("""
| Method | Formula/Parameters | Effect |
|--------|-------------------|--------|
| **Gamma (γ=0.4)** | $L' = 255(L/255)^{0.4}$ | Strong brightening |
| **Gamma (γ=0.6)** | $L' = 255(L/255)^{0.6}$ | Moderate brightening |
| **Gamma (γ=0.8)** | $L' = 255(L/255)^{0.8}$ | Mild brightening |
| **CLAHE (Clip=1.5)** | Low contrast limit | Subtle enhancement |
| **CLAHE (Clip=2.5)** | Higher contrast limit | Stronger local contrast |
| **Percentile (1%-99%)** | Clip outliers | Full dynamic range |
"""))

display(Markdown(f"✅ **Saved {len(enhancement_montage_paths)} comparison montages**"))
display(Markdown(f"📁 **Output:** `data/enhancement_comparison/`"))

In [ ]:
# ============================================================================
# SAVE MONTAGE COMPARING 4+ ENHANCEMENT RESULTS
# ============================================================================

def create_4plus_comparison_montage(img_paths, output_dir="data/comparison_montage/"):
    """Create and save montage comparing 4+ enhancement methods."""

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    display(Markdown("## 🖼️ Comparison Montage: 4+ Enhancement Methods"))

    montage_paths = []

    for i, img_path in enumerate(img_paths):
        img_bgr = imread_bgr(img_path)

        # Apply 4+ different enhancement methods
        img_gamma = apply_gamma_clahe(img_bgr, gamma=0.6, clip_limit=0)  # Gamma only
        img_clahe = apply_gamma_clahe(img_bgr, gamma=1.0, clip_limit=2.5)  # CLAHE only
        img_gamma_clahe = apply_gamma_clahe(img_bgr, gamma=0.6, clip_limit=2.5)  # Combined
        img_full = apply_final_adjustment(
            apply_unsharp_mask(
                edge_preserving_denoise(
                    apply_gamma_clahe(white_balance(img_bgr)))))  # Full pipeline

        # Create 2x3 montage (6 images)
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        # Row 1
        axes[0, 0].imshow(bgr2rgb(img_bgr))
        axes[0, 0].set_title('1. Original', fontsize=12, fontweight='bold')
        axes[0, 0].axis('off')

        axes[0, 1].imshow(bgr2rgb(img_gamma))
        axes[0, 1].set_title('2. Gamma (γ=0.6)', fontsize=12, fontweight='bold')
        axes[0, 1].axis('off')

        axes[0, 2].imshow(bgr2rgb(img_clahe))
        axes[0, 2].set_title('3. CLAHE (Clip=2.5)', fontsize=12, fontweight='bold')
        axes[0, 2].axis('off')

        # Row 2
        axes[1, 0].imshow(bgr2rgb(img_gamma_clahe))
        axes[1, 0].set_title('4. Gamma + CLAHE', fontsize=12, fontweight='bold')
        axes[1, 0].axis('off')

        axes[1, 1].imshow(bgr2rgb(white_balance(img_bgr)))
        axes[1, 1].set_title('5. White Balance', fontsize=12, fontweight='bold')
        axes[1, 1].axis('off')

        axes[1, 2].imshow(bgr2rgb(img_full))
        axes[1, 2].set_title('6. Full Pipeline', fontsize=12, fontweight='bold')
        axes[1, 2].axis('off')

        plt.suptitle(f'Enhancement Comparison - Image {i+1}', fontsize=16, fontweight='bold')
        plt.tight_layout()

        # Save montage
        montage_path = f"{output_dir}montage_{i+1}.png"
        plt.savefig(montage_path, dpi=150, bbox_inches='tight', facecolor='white')
        montage_paths.append(montage_path)
        plt.show()

        display(Markdown(f"**Image {i+1}** saved to `{montage_path}`"))

    return montage_paths

# ============================================================================
# EXECUTE
# ============================================================================
display(Markdown("---"))
display(Markdown(f"**Processing {len(img_path_array)} images...**"))
display(Markdown("""
**Methods in Montage:**
1. Original
2. Gamma Correction (γ=0.6)
3. CLAHE (Clip Limit=2.5)
4. Gamma + CLAHE Combined
5. White Balance
6. Full Enhancement Pipeline
"""))

montage_paths = create_4plus_comparison_montage(img_path_array)

display(Markdown("---"))
display(Markdown(f"✅ **Saved {len(montage_paths)} montages to** `data/comparison_montage/`"))

## Task 3: Denoising
(Starting from your best Task 2 output, compare at least two denoisers)

1. Bilateral filtering on luminance (or on each channel carefully)

In [ ]:
# ============================================================================
# TASK 3: DENOISING (25%)
# ============================================================================
# Compare at least two denoisers starting from best Task 2 output
# (a) Bilateral filtering on luminance (or on each channel carefully)
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def imread_bgr(path):
    """Read an image from a file path in BGR format."""
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    """Convert BGR image to RGB format."""
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

def to_uint8(x):
    """Clip values to [0, 255] and convert to uint8."""
    return np.clip(x, 0, 255).astype(np.uint8)

# ============================================================================
# BILATERAL FILTER IMPLEMENTATIONS
# ============================================================================

def bilateral_luminance_only(img_bgr, d=9, sigmaColor=75, sigmaSpace=75):
    """Apply bilateral filter to L channel only (preserves color better).

    Args:
        img_bgr: Input image in BGR format
        d: Diameter of pixel neighborhood (use -1 for automatic based on sigmaSpace)
        sigmaColor: Filter sigma in the color space (larger = more colors mixed)
        sigmaSpace: Filter sigma in the coordinate space (larger = farther pixels influence)

    Returns:
        Denoised image in BGR format
    """
    # Convert to LAB color space
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)

    # Apply bilateral filter only to L channel
    l_channel = img_lab[:, :, 0]
    l_denoised = cv.bilateralFilter(l_channel, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)

    # Replace L channel and convert back
    img_lab[:, :, 0] = l_denoised
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)


def bilateral_per_channel(img_bgr, d=9, sigmaColor=75, sigmaSpace=75):
    """Apply bilateral filter to each BGR channel separately.

    Args:
        img_bgr: Input image in BGR format
        d: Diameter of pixel neighborhood
        sigmaColor: Filter sigma in the color space
        sigmaSpace: Filter sigma in the coordinate space

    Returns:
        Denoised image in BGR format
    """
    # Split channels
    b, g, r = cv.split(img_bgr)

    # Apply bilateral filter to each channel
    b_denoised = cv.bilateralFilter(b, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)
    g_denoised = cv.bilateralFilter(g, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)
    r_denoised = cv.bilateralFilter(r, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)

    # Merge channels back
    return cv.merge([b_denoised, g_denoised, r_denoised])


def bilateral_full_color(img_bgr, d=9, sigmaColor=75, sigmaSpace=75):
    """Apply bilateral filter to full color image (OpenCV's native implementation).

    Args:
        img_bgr: Input image in BGR format
        d: Diameter of pixel neighborhood
        sigmaColor: Filter sigma in the color space
        sigmaSpace: Filter sigma in the coordinate space

    Returns:
        Denoised image in BGR format
    """
    return cv.bilateralFilter(img_bgr, d=d, sigmaColor=sigmaColor, sigmaSpace=sigmaSpace)


# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_bilateral_methods(img_bgr, img_index=1):
    """Compare different bilateral filtering approaches.

    Args:
        img_bgr: Input image (best output from Task 2)
        img_index: Image number for display

    Returns:
        Dictionary of denoised images
    """
    # Different parameter settings to compare
    params_mild = {'d': 5, 'sigmaColor': 50, 'sigmaSpace': 50}
    params_moderate = {'d': 9, 'sigmaColor': 75, 'sigmaSpace': 75}
    params_strong = {'d': 15, 'sigmaColor': 100, 'sigmaSpace': 100}

    results = {
        'Original': img_bgr,
        'Bilateral L-only (mild)': bilateral_luminance_only(img_bgr, **params_mild),
        'Bilateral L-only (moderate)': bilateral_luminance_only(img_bgr, **params_moderate),
        'Bilateral L-only (strong)': bilateral_luminance_only(img_bgr, **params_strong),
        'Bilateral Full Color': bilateral_full_color(img_bgr, **params_moderate),
        'Bilateral Per-Channel': bilateral_per_channel(img_bgr, **params_moderate),
    }

    # Create comparison montage (2x3 grid)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, (title, img) in enumerate(results.items()):
        axes[idx].imshow(bgr2rgb(img))
        axes[idx].set_title(title, fontsize=11)
        axes[idx].axis('off')

    plt.suptitle(f'Task 3(a): Bilateral Filter Comparison - Image {img_index}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'data/task3a_bilateral_comparison_{img_index}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results


# ============================================================================
# EXECUTE BILATERAL FILTER COMPARISON
# ============================================================================

# Define image paths (use Task 2 best output - tone_lift images)
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Use tone_lift (Gamma+CLAHE) output as starting point for denoising
tone_lift_path = "data/tone_lift/"
tone_lift_paths = [f"{tone_lift_path}gc_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
tone_lift_paths = [p for p in tone_lift_paths if os.path.exists(p)]

display(Markdown("## Task 3(a): Bilateral Filtering Comparison"))
display(Markdown("**Starting from best Task 2 output (Gamma + CLAHE)**"))

bilateral_results = {}

for i, img_path in enumerate(tone_lift_paths[:2]):  # Process first 2 images for comparison
    img = imread_bgr(img_path)
    display(Markdown(f"---\n### Image {i+1}"))
    bilateral_results[i+1] = compare_bilateral_methods(img, img_index=i+1)

# Save best bilateral results
display(Markdown("---\n### Bilateral Filter Parameters Summary"))
print("""
┌─────────────────────────────────────────────────────────────────────────────┐
│ Bilateral Filter Parameters                                                  │
├─────────────────┬───────────────────────────────────────────────────────────┤
│ Parameter       │ Effect                                                     │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ d               │ Diameter of pixel neighborhood (5-15 typical)              │
│ sigmaColor      │ Color similarity threshold (larger = more smoothing)       │
│ sigmaSpace      │ Spatial distance influence (larger = wider averaging)      │
├─────────────────┴───────────────────────────────────────────────────────────┤
│ L-only vs Full Color:                                                        │
│ • L-only: Preserves color fidelity, denoises luminance only                 │
│ • Full Color: Smooths both luminance and chrominance                        │
│ • Per-Channel: May introduce color shifts at edges                          │
└─────────────────────────────────────────────────────────────────────────────┘
""")

2. Non-local means denoising (cv.fastNlMeansDenoisingColored)

In [ ]:
# ============================================================================
# TASK 3(b): NON-LOCAL MEANS DENOISING
# ============================================================================
# cv.fastNlMeansDenoisingColored for color images
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# ============================================================================
# HELPER FUNCTIONS (if not already defined)
# ============================================================================

def imread_bgr(path):
    """Read an image from a file path in BGR format."""
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    """Convert BGR image to RGB format."""
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

# ============================================================================
# NON-LOCAL MEANS IMPLEMENTATIONS
# ============================================================================

def nlm_denoise_color(img_bgr, h=10, h_color=10, template_size=7, search_size=21):
    """Apply Non-Local Means denoising for color images.

    Args:
        img_bgr: Input image in BGR format
        h: Filter strength for luminance component
        h_color: Filter strength for color components
        template_size: Size of template patch (should be odd, 7 is default)
        search_size: Size of area where search is performed (should be odd, 21 is default)

    Returns:
        Denoised image in BGR format
    """
    # Use positional arguments: src, dst, h, hForColorComponents, templateWindowSize, searchWindowSize
    return cv.fastNlMeansDenoisingColored(img_bgr, None, h, h_color, template_size, search_size)


def nlm_denoise_luminance_only(img_bgr, h=10, template_size=7, search_size=21):
    """Apply Non-Local Means denoising to L channel only (preserves color better).

    Args:
        img_bgr: Input image in BGR format
        h: Filter strength (larger = more noise removed)
        template_size: Size of template patch
        search_size: Size of search area

    Returns:
        Denoised image in BGR format
    """
    # Convert to LAB
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)

    # Apply NLM to L channel only (use positional args)
    l_channel = img_lab[:, :, 0]
    l_denoised = cv.fastNlMeansDenoising(l_channel, None, h, template_size, search_size)

    # Replace L channel and convert back
    img_lab[:, :, 0] = l_denoised
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)


# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_nlm_methods(img_bgr, img_index=1):
    """Compare different Non-Local Means denoising settings.

    Args:
        img_bgr: Input image (best output from Task 2)
        img_index: Image number for display

    Returns:
        Dictionary of denoised images
    """
    results = {
        'Original': img_bgr,
        'NLM Color (h=5, mild)': nlm_denoise_color(img_bgr, h=5, h_color=5),
        'NLM Color (h=10, moderate)': nlm_denoise_color(img_bgr, h=10, h_color=10),
        'NLM Color (h=15, strong)': nlm_denoise_color(img_bgr, h=15, h_color=15),
        'NLM L-only (h=10)': nlm_denoise_luminance_only(img_bgr, h=10),
        'NLM Color (large search=31)': nlm_denoise_color(img_bgr, h=10, h_color=10, search_size=31),
    }

    # Create comparison montage (2x3 grid)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, (title, img) in enumerate(results.items()):
        axes[idx].imshow(bgr2rgb(img))
        axes[idx].set_title(title, fontsize=11)
        axes[idx].axis('off')

    plt.suptitle(f'Task 3(b): Non-Local Means Denoising Comparison - Image {img_index}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'data/task3b_nlm_comparison_{img_index}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results


# ============================================================================
# BILATERAL vs NLM COMPARISON
# ============================================================================

def compare_bilateral_vs_nlm(img_bgr, img_index=1):
    """Side-by-side comparison of Bilateral and Non-Local Means.

    Args:
        img_bgr: Input image
        img_index: Image number for display

    Returns:
        Dictionary of denoised images
    """
    results = {
        'Original': img_bgr,
        'Bilateral (d=9, σ=75)': cv.bilateralFilter(img_bgr, 9, 75, 75),
        'NLM Color (h=10)': nlm_denoise_color(img_bgr, h=10, h_color=10),
        'NLM Color (h=15)': nlm_denoise_color(img_bgr, h=15, h_color=15),
    }

    # Create comparison (1x4 grid)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for idx, (title, img) in enumerate(results.items()):
        axes[idx].imshow(bgr2rgb(img))
        axes[idx].set_title(title, fontsize=12)
        axes[idx].axis('off')

    plt.suptitle(f'Bilateral vs Non-Local Means - Image {img_index}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'data/task3_bilateral_vs_nlm_{img_index}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results


# ============================================================================
# EXECUTE NLM COMPARISON
# ============================================================================

# Define paths
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Use tone_lift (Gamma+CLAHE) output as starting point
tone_lift_path = "data/tone_lift/"
tone_lift_paths = [f"{tone_lift_path}gc_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
tone_lift_paths = [p for p in tone_lift_paths if os.path.exists(p)]

display(Markdown("## Task 3(b): Non-Local Means Denoising"))
display(Markdown("**Using cv.fastNlMeansDenoisingColored**"))

nlm_results = {}

for i, img_path in enumerate(tone_lift_paths[:2]):  # Process first 2 images
    img = imread_bgr(img_path)
    display(Markdown(f"---\n### Image {i+1}: NLM Parameter Comparison"))
    nlm_results[i+1] = compare_nlm_methods(img, img_index=i+1)

    display(Markdown(f"### Image {i+1}: Bilateral vs NLM"))
    compare_bilateral_vs_nlm(img, img_index=i+1)

display(Markdown("---\n### Non-Local Means Parameters Summary"))
print("""
NLM vs Bilateral:
• NLM: Better at preserving edges and textures, but SLOWER
• Bilateral: Faster, good for real-time, may over-smooth textures
• NLM finds similar patches across the image (non-local averaging)
• Bilateral only considers local spatial + color similarity
""")

3. (Optional) Median filter for salt-and-pepper artifacts

In [ ]:
# ============================================================================
# TASK 3(c): MEDIAN FILTER FOR SALT-AND-PEPPER ARTIFACTS (Optional)
# ============================================================================
# Median filter is particularly effective for impulse noise (salt-and-pepper)
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def imread_bgr(path):
    """Read an image from a file path in BGR format."""
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    """Convert BGR image to RGB format."""
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

# ============================================================================
# MEDIAN FILTER IMPLEMENTATIONS
# ============================================================================

def median_filter_color(img_bgr, ksize=3):
    """Apply median filter to full color image.

    Args:
        img_bgr: Input image in BGR format
        ksize: Kernel size (must be odd: 3, 5, 7, etc.)

    Returns:
        Filtered image in BGR format
    """
    return cv.medianBlur(img_bgr, ksize)


def median_filter_luminance_only(img_bgr, ksize=3):
    """Apply median filter to L channel only (preserves color).

    Args:
        img_bgr: Input image in BGR format
        ksize: Kernel size (must be odd)

    Returns:
        Filtered image in BGR format
    """
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]
    l_filtered = cv.medianBlur(l_channel, ksize)
    img_lab[:, :, 0] = l_filtered
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)


def median_filter_per_channel(img_bgr, ksize=3):
    """Apply median filter to each channel separately.

    Args:
        img_bgr: Input image in BGR format
        ksize: Kernel size (must be odd)

    Returns:
        Filtered image in BGR format
    """
    b, g, r = cv.split(img_bgr)
    b_filtered = cv.medianBlur(b, ksize)
    g_filtered = cv.medianBlur(g, ksize)
    r_filtered = cv.medianBlur(r, ksize)
    return cv.merge([b_filtered, g_filtered, r_filtered])


def add_salt_and_pepper_noise(img, amount=0.02):
    """Add salt-and-pepper noise to an image for testing.

    Args:
        img: Input image
        amount: Proportion of pixels to corrupt (0.0 to 1.0)

    Returns:
        Noisy image
    """
    noisy = img.copy()
    h, w = img.shape[:2]
    num_pixels = int(amount * h * w)

    # Salt (white pixels)
    for _ in range(num_pixels // 2):
        y = np.random.randint(0, h)
        x = np.random.randint(0, w)
        noisy[y, x] = 255

    # Pepper (black pixels)
    for _ in range(num_pixels // 2):
        y = np.random.randint(0, h)
        x = np.random.randint(0, w)
        noisy[y, x] = 0

    return noisy


# ============================================================================
# COMPARISON FUNCTION
# ============================================================================

def compare_median_filters(img_bgr, img_index=1, add_noise=True):
    """Compare median filter with different kernel sizes.

    Args:
        img_bgr: Input image
        img_index: Image number for display
        add_noise: Whether to add salt-and-pepper noise for demonstration

    Returns:
        Dictionary of filtered images
    """
    # Optionally add noise for demonstration
    if add_noise:
        test_img = add_salt_and_pepper_noise(img_bgr, amount=0.02)
        noise_label = " (with S&P noise)"
    else:
        test_img = img_bgr
        noise_label = ""

    results = {
        f'Original{noise_label}': test_img,
        'Median 3×3': median_filter_color(test_img, ksize=3),
        'Median 5×5': median_filter_color(test_img, ksize=5),
        'Median 7×7': median_filter_color(test_img, ksize=7),
        'Median L-only 3×3': median_filter_luminance_only(test_img, ksize=3),
        'Median L-only 5×5': median_filter_luminance_only(test_img, ksize=5),
    }

    # Create comparison montage (2x3 grid)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, (title, img) in enumerate(results.items()):
        axes[idx].imshow(bgr2rgb(img))
        axes[idx].set_title(title, fontsize=11)
        axes[idx].axis('off')

    plt.suptitle(f'Task 3(c): Median Filter Comparison - Image {img_index}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'data/task3c_median_comparison_{img_index}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results


def compare_all_denoisers(img_bgr, img_index=1):
    """Compare all denoising methods: Bilateral, NLM, and Median.

    Args:
        img_bgr: Input image
        img_index: Image number for display

    Returns:
        Dictionary of denoised images
    """
    results = {
        'Original': img_bgr,
        'Bilateral (d=9)': cv.bilateralFilter(img_bgr, 9, 75, 75),
        'NLM (h=10)': cv.fastNlMeansDenoisingColored(img_bgr, None, 10, 10, 7, 21),
        'Median 3×3': median_filter_color(img_bgr, ksize=3),
        'Median 5×5': median_filter_color(img_bgr, ksize=5),
    }

    # Create comparison (1x5 grid)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    for idx, (title, img) in enumerate(results.items()):
        axes[idx].imshow(bgr2rgb(img))
        axes[idx].set_title(title, fontsize=11)
        axes[idx].axis('off')

    plt.suptitle(f'All Denoisers Comparison - Image {img_index}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'data/task3_all_denoisers_{img_index}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return results


# ============================================================================
# EXECUTE MEDIAN FILTER COMPARISON
# ============================================================================

# Define paths
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Use tone_lift (Gamma+CLAHE) output as starting point
tone_lift_path = "data/tone_lift/"
tone_lift_paths = [f"{tone_lift_path}gc_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
tone_lift_paths = [p for p in tone_lift_paths if os.path.exists(p)]

display(Markdown("## Task 3(c): Median Filter for Salt-and-Pepper Artifacts (Optional)"))

median_results = {}

for i, img_path in enumerate(tone_lift_paths[:2]):  # Process first 2 images
    img = imread_bgr(img_path)

    # Show median filter with artificial S&P noise
    display(Markdown(f"---\n### Image {i+1}: Median Filter with Salt-and-Pepper Noise"))
    median_results[i+1] = compare_median_filters(img, img_index=i+1, add_noise=True)

    # Compare all denoisers on original (no added noise)
    display(Markdown(f"### Image {i+1}: All Denoisers Comparison (no added noise)"))
    compare_all_denoisers(img, img_index=i+1)

# Summary
display(Markdown("---\n### Denoising Methods Summary"))
print("\n"*3)
print("""
┌──────────────────────────────────────────────────────────────────────────────┐
│ DENOISING METHODS COMPARISON                                                 │
├──────────────────┬───────────────────────────────────────────────────────────┤
│ Method           │ Best For                                                  │
├──────────────────┼───────────────────────────────────────────────────────────┤
│ Bilateral        │ Gaussian noise, edge preservation, real-time applications │
│ NLM              │ General noise, texture preservation, quality-critical     │
│ Median           │ Salt-and-pepper (impulse) noise, hot/dead pixels          │
├──────────────────┴───────────────────────────────────────────────────────────┤
│ Median Filter Properties:                                                    │
│ • Replaces each pixel with median of neighborhood                            │
│ • Excellent at removing outliers (impulse noise)                             │
│ • Preserves edges better than mean/Gaussian blur                             │
│ • 3×3: Minimal smoothing, good for mild noise                                │
│ • 5×5: Moderate smoothing, removes larger artifacts                          │
│ • 7×7+: Strong smoothing, may lose fine details                              │
└──────────────────────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ============================================================================
# TASK 3: SAVE DENOISING COMPARISON MONTAGE
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def imread_bgr(path):
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

# ============================================================================
# DENOISING FUNCTIONS
# ============================================================================

def bilateral_denoise(img_bgr, d=9, sigmaColor=75, sigmaSpace=75):
    return cv.bilateralFilter(img_bgr, d, sigmaColor, sigmaSpace)

def nlm_denoise(img_bgr, h=10, h_color=10, template_size=7, search_size=21):
    return cv.fastNlMeansDenoisingColored(img_bgr, None, h, h_color, template_size, search_size)

def median_denoise(img_bgr, ksize=5):
    return cv.medianBlur(img_bgr, ksize)

# ============================================================================
# CREATE AND SAVE DENOISING COMPARISON MONTAGE
# ============================================================================

def create_denoise_comparison_montage(img_bgr, img_index=1, save_path="data/"):
    """Create and save a denoising comparison montage.

    Args:
        img_bgr: Input image (from Task 2 best output)
        img_index: Image number
        save_path: Directory to save montage
    """
    # Apply different denoisers
    bilateral = bilateral_denoise(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
    nlm = nlm_denoise(img_bgr, h=10, h_color=10)
    median = median_denoise(img_bgr, ksize=5)

    # Create montage (2x2 grid)
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    images = [
        ('Original (after Gamma+CLAHE)', img_bgr),
        ('Bilateral Filter (d=9, σ=75)', bilateral),
        ('Non-Local Means (h=10)', nlm),
        ('Median Filter (5×5)', median),
    ]

    for idx, (title, img) in enumerate(images):
        ax = axes[idx // 2, idx % 2]
        ax.imshow(bgr2rgb(img))
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f'Denoising Comparison - Image {img_index}', fontsize=16, fontweight='bold')
    plt.tight_layout()

    # Save montage
    montage_path = f'{save_path}denoise_compare.png'
    plt.savefig(montage_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {montage_path}")
    return {'original': img_bgr, 'bilateral': bilateral, 'nlm': nlm, 'median': median}


def create_edge_comparison_crop(img_bgr, img_index=1, save_path="data/"):
    """Create zoomed crop comparison to show edge preservation.

    Args:
        img_bgr: Input image
        img_index: Image number
        save_path: Directory to save comparison
    """
    # Apply denoisers
    bilateral = bilateral_denoise(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
    nlm = nlm_denoise(img_bgr, h=10, h_color=10)
    median = median_denoise(img_bgr, ksize=5)

    # Get image dimensions and define crop region (center-ish, 200x200 pixels)
    h, w = img_bgr.shape[:2]
    crop_size = 200
    y_start = h // 3
    x_start = w // 3
    y_end = y_start + crop_size
    x_end = x_start + crop_size

    # Ensure crop is within bounds
    y_end = min(y_end, h)
    x_end = min(x_end, w)

    # Extract crops
    crops = {
        'Original': img_bgr[y_start:y_end, x_start:x_end],
        'Bilateral': bilateral[y_start:y_end, x_start:x_end],
        'NLM': nlm[y_start:y_end, x_start:x_end],
        'Median': median[y_start:y_end, x_start:x_end],
    }

    # Create zoomed crop comparison (1x4 grid)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    for idx, (title, crop) in enumerate(crops.items()):
        axes[idx].imshow(bgr2rgb(crop))
        axes[idx].set_title(f'{title}\n(Zoomed Crop)', fontsize=11)
        axes[idx].axis('off')

    plt.suptitle(f'Edge Preservation Comparison (Zoomed) - Image {img_index}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save zoomed comparison
    crop_path = f'{save_path}denoise_edge_crop_{img_index}.png'
    plt.savefig(crop_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {crop_path}")
    return crops


# ============================================================================
# EXECUTE
# ============================================================================

# Define paths
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Use tone_lift (Gamma+CLAHE) output as starting point
tone_lift_path = "data/tone_lift/"
tone_lift_paths = [f"{tone_lift_path}gc_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
tone_lift_paths = [p for p in tone_lift_paths if os.path.exists(p)]

display(Markdown("## Task 3: Denoising Comparison Montage"))

if tone_lift_paths:
    # Use first image for main montage
    img = imread_bgr(tone_lift_paths[0])

    # Create and save main comparison montage
    display(Markdown("### Denoising Methods Comparison"))
    denoise_results = create_denoise_comparison_montage(img, img_index=1, save_path="data/")

    # Create zoomed edge comparison
    display(Markdown("### Edge Preservation Analysis (Zoomed Crop)"))
    edge_crops = create_edge_comparison_crop(img, img_index=1, save_path="data/")
else:
    print("No tone_lift images found. Run Task 2 first.")

# ============================================================================
# TASK 3 QUESTIONS (for report)
# ============================================================================

display(Markdown("---"))
display(Markdown("## Task 3 Questions"))

display(Markdown("""
### 1. Why does contrast enhancement tend to make noise more visible?

**Answer:**

Contrast enhancement (Gamma correction, CLAHE, histogram stretching) **amplifies the difference between pixel intensities**. This has two effects on noise:

1. **Shadow noise amplification**: In dark regions, the signal-to-noise ratio (SNR) is low. When gamma (γ < 1) lifts shadows, it amplifies both the signal AND the noise equally. Small random variations that were invisible in dark areas become visible.

2. **CLAHE local amplification**: CLAHE enhances local contrast by redistributing histogram bins. In relatively uniform areas (sky, walls), small noise variations get stretched across a wider intensity range, making them more prominent.

3. **Mathematical explanation**: If original noise has variance σ², after gamma correction with γ < 1, the noise variance increases because:
   - Derivative of gamma function: d/dL[L^γ] = γ·L^(γ-1)
   - For dark pixels (small L) and γ < 1, this derivative is LARGE
   - Large derivative = large amplification of small variations

**This is why denoising should come AFTER contrast enhancement in the pipeline.**

---

### 2. Which denoiser preserved edges best on your image? Provide evidence (zoomed crop).

**Answer:**

Based on the zoomed crop comparison, **Non-Local Means (NLM)** typically preserves edges best because:

| Denoiser | Edge Preservation | Observations |
|----------|-------------------|--------------|
| **Bilateral** | Good | Preserves strong edges, may blur fine textures |
| **NLM** | Best | Preserves edges AND textures, finds similar patches globally |
| **Median** | Moderate | Good for impulse noise, can round corners |

**Evidence from zoomed crops:**
- NLM maintains sharp transitions at object boundaries
- NLM preserves fine texture details (e.g., fabric, foliage)
- Bilateral may create slight "cartoon" effect on gradients
- Median can blur fine edge details

**Why NLM is better for edges:**
- Uses patch-based similarity (7×7 patches by default)
- Searches for similar patches across entire image (21×21 search window)
- Averages only truly similar regions, preserving unique edge patterns
- Bilateral only considers local spatial+color distance, not pattern similarity

*See the saved zoomed crop image (denoise_edge_crop_1.png) for visual evidence.*
"""))

## Task 4: Sharpening with Artifact Control (20%)

Implement unsharp masking:
blur = Gσ ∗ f, g = f + α(f − blur).
Try at least three (σ, α) pairs (e.g. σ ∈ {1, 2}, α ∈ {0.6, 1.0, 1.4}). Save a grid and your final choice.

In [ ]:
# ============================================================================
# TASK 4: SHARPENING (15%)
# ============================================================================
# Implement unsharp masking: blur = Gσ * f, g = f + α(f - blur)
# Try at least three (σ, α) pairs
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def imread_bgr(path):
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

def to_uint8(x):
    return np.clip(x, 0, 255).astype(np.uint8)

# ============================================================================
# UNSHARP MASKING IMPLEMENTATION
# ============================================================================

def unsharp_mask(img_bgr, sigma=1.0, alpha=1.0, apply_to_luminance=True):
    """Apply unsharp masking: g = f + α(f - blur)

    Args:
        img_bgr: Input image in BGR format
        sigma: Gaussian blur sigma (controls blur radius)
        alpha: Sharpening strength (higher = more sharpening)
        apply_to_luminance: If True, apply only to L channel (preserves colors)

    Returns:
        Sharpened image in BGR format
    """
    if apply_to_luminance:
        # Apply to L channel only (better color preservation)
        img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
        l_channel = img_lab[:, :, 0].astype(np.float32)

        # blur = Gσ * f
        blur = cv.GaussianBlur(l_channel, (0, 0), sigmaX=sigma, sigmaY=sigma)

        # g = f + α(f - blur)
        detail = l_channel - blur
        sharpened = l_channel + alpha * detail

        # Clip and convert back
        img_lab[:, :, 0] = to_uint8(sharpened)
        return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)
    else:
        # Apply to full color image
        img_float = img_bgr.astype(np.float32)
        blur = cv.GaussianBlur(img_float, (0, 0), sigmaX=sigma, sigmaY=sigma)
        detail = img_float - blur
        sharpened = img_float + alpha * detail
        return to_uint8(sharpened)


def unsharp_mask_opencv(img_bgr, sigma=1.0, alpha=1.0):
    """Alternative implementation using OpenCV's addWeighted.

    g = f + α(f - blur) = (1 + α)f - α*blur = addWeighted(f, 1+α, blur, -α, 0)
    """
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    l_channel = img_lab[:, :, 0]

    blur = cv.GaussianBlur(l_channel, (0, 0), sigmaX=sigma, sigmaY=sigma)

    # Using addWeighted: dst = src1*alpha + src2*beta + gamma
    # g = (1+α)*f + (-α)*blur = f + α*(f - blur)
    sharpened = cv.addWeighted(l_channel, 1.0 + alpha, blur, -alpha, 0)

    img_lab[:, :, 0] = sharpened
    return cv.cvtColor(img_lab, cv.COLOR_LAB2BGR)


# ============================================================================
# PARAMETER GRID COMPARISON
# ============================================================================

def create_sharpening_grid(img_bgr, img_index=1, save_path="data/"):
    """Create a grid comparing different (σ, α) combinations.

    Args:
        img_bgr: Input image (denoised output from Task 3)
        img_index: Image number
        save_path: Directory to save grid

    Returns:
        Dictionary of sharpened images and best parameters
    """
    # Define parameter combinations to test
    sigma_values = [1.0, 2.0, 3.0]
    alpha_values = [0.6, 1.0, 1.4]

    # Create results dictionary
    results = {}

    # Create grid figure (3 sigma x 3 alpha = 9 images)
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))

    for i, sigma in enumerate(sigma_values):
        for j, alpha in enumerate(alpha_values):
            sharpened = unsharp_mask(img_bgr, sigma=sigma, alpha=alpha)
            key = f"σ={sigma}, α={alpha}"
            results[key] = sharpened

            axes[i, j].imshow(bgr2rgb(sharpened))
            axes[i, j].set_title(f"σ={sigma}, α={alpha}", fontsize=11, fontweight='bold')
            axes[i, j].axis('off')

    plt.suptitle(f'Unsharp Masking Parameter Grid - Image {img_index}\ng = f + α(f - Gσ*f)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save grid
    grid_path = f'{save_path}sharpen_grid_{img_index}.png'
    plt.savefig(grid_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {grid_path}")
    return results


def show_final_choice(img_bgr, sigma=1.5, alpha=1.0, img_index=1, save_path="data/"):
    """Show original vs final sharpened choice.

    Args:
        img_bgr: Input image
        sigma: Chosen sigma value
        alpha: Chosen alpha value
        img_index: Image number
        save_path: Directory to save

    Returns:
        Sharpened image
    """
    sharpened = unsharp_mask(img_bgr, sigma=sigma, alpha=alpha)

    # Create side-by-side comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original (denoised input)
    axes[0].imshow(bgr2rgb(img_bgr))
    axes[0].set_title('Input (Denoised)', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # Sharpened
    axes[1].imshow(bgr2rgb(sharpened))
    axes[1].set_title(f'Sharpened (σ={sigma}, α={alpha})', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    # Difference (enhanced for visibility)
    diff = cv.absdiff(img_bgr, sharpened)
    diff_enhanced = cv.convertScaleAbs(diff, alpha=5.0)  # Amplify for visibility
    axes[2].imshow(bgr2rgb(diff_enhanced))
    axes[2].set_title('Difference (5× enhanced)', fontsize=12, fontweight='bold')
    axes[2].axis('off')

    plt.suptitle(f'Final Sharpening Choice - Image {img_index}', fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save final choice
    final_path = f'{save_path}sharpen_final_{img_index}.png'
    plt.savefig(final_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {final_path}")

    # Also save the sharpened image itself
    sharpened_img_path = f'{save_path}sharpened/sharp_{img_index}.jpg'
    os.makedirs(f'{save_path}sharpened/', exist_ok=True)
    cv.imwrite(sharpened_img_path, sharpened)
    print(f"Saved sharpened image: {sharpened_img_path}")

    return sharpened


def create_zoomed_sharpening_comparison(img_bgr, sigma=1.5, alpha=1.0, img_index=1, save_path="data/"):
    """Create zoomed crop to show sharpening effect on edges.

    Args:
        img_bgr: Input image
        sigma: Sigma value
        alpha: Alpha value
        img_index: Image number
        save_path: Directory to save
    """
    sharpened = unsharp_mask(img_bgr, sigma=sigma, alpha=alpha)

    # Define crop region
    h, w = img_bgr.shape[:2]
    crop_size = 250
    y_start = h // 3
    x_start = w // 3
    y_end = min(y_start + crop_size, h)
    x_end = min(x_start + crop_size, w)

    # Extract crops
    orig_crop = img_bgr[y_start:y_end, x_start:x_end]
    sharp_crop = sharpened[y_start:y_end, x_start:x_end]

    # Create comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(bgr2rgb(orig_crop))
    axes[0].set_title('Before Sharpening (Zoomed)', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(bgr2rgb(sharp_crop))
    axes[1].set_title(f'After Sharpening σ={sigma}, α={alpha} (Zoomed)', fontsize=12, fontweight='bold')
    axes[1].axis('off')

    plt.suptitle(f'Sharpening Detail Comparison - Image {img_index}', fontsize=14, fontweight='bold')
    plt.tight_layout()

    zoom_path = f'{save_path}sharpen_zoom_{img_index}.png'
    plt.savefig(zoom_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {zoom_path}")


# ============================================================================
# EXECUTE SHARPENING COMPARISON
# ============================================================================

# Define paths
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Use denoised output as starting point (or tone_lift if denoised not available)
denoised_path = "data/denoised/"
denoised_paths = [f"{denoised_path}ep_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
denoised_paths = [p for p in denoised_paths if os.path.exists(p)]

# Fallback to tone_lift if denoised not found
if not denoised_paths:
    tone_lift_path = "data/tone_lift/"
    denoised_paths = [f"{tone_lift_path}gc_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
    denoised_paths = [p for p in denoised_paths if os.path.exists(p)]

display(Markdown("## Task 4: Sharpening (Unsharp Masking)"))
display(Markdown("**Formula:** g = f + α(f - Gσ*f)"))
display(Markdown("**Parameters:** σ ∈ {1, 2, 3}, α ∈ {0.6, 1.0, 1.4}"))

sharpening_results = {}

if denoised_paths:
    for i, img_path in enumerate(denoised_paths[:2]):  # Process first 2 images
        img = imread_bgr(img_path)

        display(Markdown(f"---\n### Image {i+1}: Parameter Grid"))
        sharpening_results[i+1] = create_sharpening_grid(img, img_index=i+1, save_path="data/")

        # Show final choice (σ=1.5, α=1.0 is often a good balance)
        display(Markdown(f"### Image {i+1}: Final Choice"))
        final_sharpened = show_final_choice(img, sigma=1.5, alpha=1.0, img_index=i+1, save_path="data/")

        # Zoomed comparison
        display(Markdown(f"### Image {i+1}: Zoomed Detail"))
        create_zoomed_sharpening_comparison(img, sigma=1.5, alpha=1.0, img_index=i+1, save_path="data/")
else:
    print("No denoised images found. Run Task 3 first.")

# ============================================================================
# PARAMETER SELECTION RATIONALE
# ============================================================================

display(Markdown("---"))
display(Markdown("### Unsharp Masking Parameter Guide"))
display(Markdown("""
| Parameter | Effect | Recommended Range |
|-----------|--------|-------------------|
| **σ (sigma)** | Blur radius - controls scale of detail enhancement | 1.0 - 3.0 |
| | Small σ (1.0): Enhances fine details, edges | |
| | Large σ (3.0): Enhances larger features, halos possible | |
| **α (alpha)** | Sharpening strength | 0.5 - 1.5 |
| | Small α (0.6): Subtle sharpening | |
| | Large α (1.4+): Strong sharpening, may cause artifacts | |

**Final Choice Rationale: σ=1.5, α=1.0**
- σ=1.5: Good balance between fine edges and medium details
- α=1.0: Moderate sharpening without introducing halos
- Applied to L-channel only: Preserves color integrity
- Suitable for night images where over-sharpening amplifies remaining noise
"""))

In [ ]:
# ============================================================================
# TASK 4 QUESTIONS (for report)
# ============================================================================

from IPython.display import Markdown, display

display(Markdown("---"))
display(Markdown("## Task 4 Questions"))

display(Markdown("""
### 1. What happens as σ increases? (What spatial scales are boosted?)

**Answer:**

As σ (sigma) increases in the unsharp mask formula `g = f + α(f - Gσ*f)`:

| σ Value | Blur Radius | Spatial Scales Boosted | Effect |
|---------|-------------|------------------------|--------|
| **σ = 1** | ~3 pixels | Fine edges, small details | Sharpens text, thin lines, fine textures |
| **σ = 2** | ~6 pixels | Medium features | Sharpens object edges, medium textures |
| **σ = 3+** | ~9+ pixels | Large features, coarse structures | Enhances overall contrast, broad transitions |

**Mathematical Explanation:**

The Gaussian blur kernel size is approximately **6σ** (covers 99.7% of the distribution). The detail signal `(f - Gσ*f)` contains frequencies that are:
- **Removed** by the blur (high frequencies smaller than σ)
- **Preserved** after subtraction (these get boosted)

**As σ increases:**
1. The blur becomes wider, averaging over more pixels
2. Finer details get blurred away (included in the "blur" term)
3. Only **coarser/larger features** remain in the difference `(f - blur)`
4. These larger-scale features get amplified by α

**Visual Effect:**
- **Small σ (1.0)**: Enhances fine edges like hair, text, fabric texture
- **Medium σ (2.0)**: Enhances object boundaries, facial features
- **Large σ (3.0+)**: Creates local contrast enhancement, similar to CLAHE effect, can cause "halo" artifacts around high-contrast edges

---

### 2. What visible artifacts occur if α is too large?

**Answer:**

When α (sharpening strength) is too large, several artifacts appear:

| Artifact | Description | Cause |
|----------|-------------|-------|
| **Halo/Ringing** | Bright or dark bands along high-contrast edges | Over-amplified `(f - blur)` creates overshoot at transitions |
| **Noise Amplification** | Grainy appearance, especially in uniform areas | Noise is part of "detail" signal and gets boosted |
| **Clipping/Saturation** | Pure white or black regions near edges | Values exceed 0-255 range after amplification |
| **Unnatural Edges** | Edges look artificially enhanced, "crunchy" | Excessive edge contrast breaks natural appearance |

**Example with α values:**

```
α = 0.6  → Subtle enhancement, natural look
α = 1.0  → Moderate sharpening, slight halos possible
α = 1.4  → Strong sharpening, visible halos on high-contrast edges
α = 2.0+ → Severe artifacts, image looks over-processed
```

**Mathematical Reason:**

In the formula `g = f + α(f - blur)`:
- The detail term `(f - blur)` can be positive or negative
- At a bright→dark edge: detail is positive on bright side, negative on dark side
- Multiplying by large α amplifies this difference
- Result: **white overshoot** on bright side, **black undershoot** on dark side = **halo**

**Visual Example of Halos:**
```
Original edge:    ████████░░░░░░░░
α = 1.0:          █████████░░░░░░░  (slight enhancement)
α = 2.0:          ██████████▓░░░░░  (visible halo - bright band before dark)
α = 3.0:          ███████████▓▒░░░  (severe halo - obvious artifact)
```

**Best Practice:**
- Keep α ≤ 1.5 for most images
- For noisy images (like enhanced night photos), use α ≤ 1.0
- Always check zoomed crops for halo artifacts before finalizing
"""))

## Task 4 Questions

### 1. What happens as σ increases? (What spatial scales are boosted?)

| σ Value | Spatial Scales Boosted | Effect |
|---------|------------------------|--------|
| **Small σ (1)** | Fine details | Sharpens pixels, thin edges, text, fine textures |
| **Medium σ (2)** | Medium features | Sharpens object edges, facial features |
| **Large σ (3+)** | Coarse structures | Enhances overall contrast, can cause halos |

**Why this happens:**
- The blur kernel size is ~6σ pixels
- Larger σ = wider blur = only coarse features remain in `(f - blur)`
- These coarse features get amplified by α

---

### 2. What visible artifacts occur if α is too large?

| Artifact | Description |
|----------|-------------|
| **Halos/Ringing** | Bright or dark bands along high-contrast edges |
| **Noise Amplification** | Grainy appearance, especially in uniform areas |
| **Clipping** | Pure white or black regions near edges |
| **Unnatural Look** | Edges appear artificially enhanced, "crunchy" |

**Recommended α values:**
- `α = 0.6` → Subtle, natural enhancement
- `α = 1.0` → Moderate sharpening
- `α = 1.4` → Strong sharpening, halos may appear
- `α > 2.0` → Severe artifacts, over-processed look

---

### Best Practices

- **For night images:** Use `σ = 1.0–1.5` and `α ≤ 1.0` (noise is already amplified)
- **For clean images:** Can use `α` up to 1.5
- **Always check:** Zoomed crops for halo artifacts before finalizing

In [ ]:
# ============================================================================
# TASK 5: OBJECTIVE EVALUATION (15%)
# ============================================================================
# Compute metrics before vs after:
# 1. Sharpness proxy: Variance of Laplacian
# 2. Global contrast: Standard deviation of luminance
# 3. Shadow visibility: Fraction of pixels below threshold
# ============================================================================

import os
import cv2 as cv
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import Markdown, display
import pandas as pd

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def imread_bgr(path):
    img = cv.imread(path, cv.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read: {path}")
    return img

def bgr2rgb(img):
    return cv.cvtColor(img, cv.COLOR_BGR2RGB)

def get_luminance(img_bgr):
    """Extract luminance channel (L from LAB)."""
    img_lab = cv.cvtColor(img_bgr, cv.COLOR_BGR2LAB)
    return img_lab[:, :, 0]

# ============================================================================
# METRIC FUNCTIONS
# ============================================================================

def variance_of_laplacian(img_bgr):
    """Compute sharpness proxy: Variance of Laplacian.

    S = Var(∇²I)

    Higher values indicate sharper images with more edge content.

    Args:
        img_bgr: Input image in BGR format

    Returns:
        Variance of Laplacian (float)
    """
    # Convert to grayscale (or use L channel)
    gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

    # Compute Laplacian (second-order derivative)
    laplacian = cv.Laplacian(gray, cv.CV_64F, ksize=3)

    # Return variance
    return float(laplacian.var())


def global_contrast(img_bgr):
    """Compute global contrast: Standard deviation of luminance.

    Higher values indicate more contrast.

    Args:
        img_bgr: Input image in BGR format

    Returns:
        Standard deviation of luminance (float)
    """
    luminance = get_luminance(img_bgr)
    return float(luminance.std())


def shadow_fraction(img_bgr, threshold=20):
    """Compute shadow visibility: Fraction of pixels below threshold.

    Lower values after enhancement indicate better shadow recovery.

    Args:
        img_bgr: Input image in BGR format
        threshold: Pixel intensity threshold (default 20)

    Returns:
        Fraction of pixels below threshold (float, 0-1)
    """
    luminance = get_luminance(img_bgr)
    below_threshold = luminance < threshold
    return float(np.mean(below_threshold))


def compute_all_metrics(img_bgr):
    """Compute all three metrics for an image.

    Args:
        img_bgr: Input image in BGR format

    Returns:
        Dictionary with all metrics
    """
    return {
        'Sharpness (VoL)': variance_of_laplacian(img_bgr),
        'Contrast (Std)': global_contrast(img_bgr),
        'Shadow Fraction': shadow_fraction(img_bgr, threshold=20),
    }


# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_pipeline(original_path, enhanced_path, img_index=1):
    """Evaluate original vs enhanced image with all metrics.

    Args:
        original_path: Path to original image
        enhanced_path: Path to enhanced image
        img_index: Image number

    Returns:
        Dictionary with before/after metrics and improvements
    """
    original = imread_bgr(original_path)
    enhanced = imread_bgr(enhanced_path)

    # Compute metrics
    metrics_before = compute_all_metrics(original)
    metrics_after = compute_all_metrics(enhanced)

    # Calculate improvements
    improvements = {}
    for key in metrics_before:
        before = metrics_before[key]
        after = metrics_after[key]

        if key == 'Shadow Fraction':
            # For shadows, lower is better (negative change is improvement)
            change = before - after
            pct = (change / before * 100) if before > 0 else 0
            improvements[key] = {'change': -change, 'pct': -pct, 'improved': change > 0}
        else:
            # For sharpness and contrast, higher is better
            change = after - before
            pct = (change / before * 100) if before > 0 else 0
            improvements[key] = {'change': change, 'pct': pct, 'improved': change > 0}

    return {
        'before': metrics_before,
        'after': metrics_after,
        'improvements': improvements,
        'original': original,
        'enhanced': enhanced
    }


def create_evaluation_summary(results_list, save_path="data/"):
    """Create summary table and visualization of all evaluations.

    Args:
        results_list: List of evaluation results
        save_path: Directory to save outputs
    """
    # Build summary data
    summary_data = []

    for i, results in enumerate(results_list):
        row = {'Image': i + 1}
        for metric in results['before']:
            row[f'{metric} (Before)'] = results['before'][metric]
            row[f'{metric} (After)'] = results['after'][metric]
            row[f'{metric} (Change %)'] = results['improvements'][metric]['pct']
        summary_data.append(row)

    # Create DataFrame
    df = pd.DataFrame(summary_data)

    # Display table
    display(Markdown("### Metrics Summary Table"))
    display(df.round(2))

    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    metrics = ['Sharpness (VoL)', 'Contrast (Std)', 'Shadow Fraction']
    colors_before = '#3498db'  # Blue
    colors_after = '#2ecc71'   # Green

    for idx, metric in enumerate(metrics):
        before_vals = [r['before'][metric] for r in results_list]
        after_vals = [r['after'][metric] for r in results_list]

        x = np.arange(len(results_list))
        width = 0.35

        axes[idx].bar(x - width/2, before_vals, width, label='Before', color=colors_before)
        axes[idx].bar(x + width/2, after_vals, width, label='After', color=colors_after)

        axes[idx].set_xlabel('Image')
        axes[idx].set_ylabel(metric)
        axes[idx].set_title(metric)
        axes[idx].set_xticks(x)
        axes[idx].set_xticklabels([f'Img {i+1}' for i in range(len(results_list))])
        axes[idx].legend()

        # Add improvement annotation
        for i, (b, a) in enumerate(zip(before_vals, after_vals)):
            if metric == 'Shadow Fraction':
                pct = ((b - a) / b * 100) if b > 0 else 0
                color = 'green' if a < b else 'red'
            else:
                pct = ((a - b) / b * 100) if b > 0 else 0
                color = 'green' if a > b else 'red'
            axes[idx].annotate(f'{pct:+.1f}%', xy=(i, max(b, a)),
                              ha='center', va='bottom', fontsize=9, color=color)

    plt.suptitle('Objective Evaluation: Before vs After Enhancement', fontsize=14, fontweight='bold')
    plt.tight_layout()

    # Save figure
    eval_path = f'{save_path}evaluation_metrics.png'
    plt.savefig(eval_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {eval_path}")

    return df


def show_before_after_with_metrics(results, img_index=1, save_path="data/"):
    """Show before/after comparison with metrics overlay.

    Args:
        results: Evaluation results dictionary
        img_index: Image number
        save_path: Directory to save
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Before
    axes[0].imshow(bgr2rgb(results['original']))
    metrics_text = '\n'.join([f"{k}: {v:.2f}" for k, v in results['before'].items()])
    axes[0].set_title(f"Original\n{metrics_text}", fontsize=10)
    axes[0].axis('off')

    # After
    axes[1].imshow(bgr2rgb(results['enhanced']))
    metrics_text = '\n'.join([f"{k}: {v:.2f}" for k, v in results['after'].items()])
    improvements_text = '\n'.join([
        f"{k}: {v['pct']:+.1f}%" for k, v in results['improvements'].items()
    ])
    axes[1].set_title(f"Enhanced\n{metrics_text}", fontsize=10)
    axes[1].axis('off')

    plt.suptitle(f'Image {img_index}: Before vs After with Metrics', fontsize=14, fontweight='bold')
    plt.tight_layout()

    compare_path = f'{save_path}evaluation_compare_{img_index}.png'
    plt.savefig(compare_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()

    print(f"Saved: {compare_path}")


# ============================================================================
# EXECUTE EVALUATION
# ============================================================================

# Define paths
img_path = "data/"
img_path_array = [f"{img_path}{i}.jpg" for i in range(1, 7)]
img_path_array = [p for p in img_path_array if cv.imread(p) is not None]

# Final enhanced images (from pipeline)
final_path = "data/contrast_saturation/"
final_paths = [f"{final_path}cs_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
final_paths = [p for p in final_paths if os.path.exists(p)]

# Fallback to sharpened if final not found
if not final_paths:
    final_path = "data/sharpened/"
    final_paths = [f"{final_path}sharp_{i}.jpg" for i in range(1, len(img_path_array) + 1)]
    final_paths = [p for p in final_paths if os.path.exists(p)]

display(Markdown("## Task 5: Objective Evaluation"))
display(Markdown("""
**Metrics computed:**
1. **Sharpness (VoL)**: Variance of Laplacian - higher = sharper
2. **Contrast (Std)**: Standard deviation of luminance - higher = more contrast
3. **Shadow Fraction**: Pixels below threshold 20 - lower = better shadow recovery
"""))

evaluation_results = []

if img_path_array and final_paths:
    num_images = min(len(img_path_array), len(final_paths))

    for i in range(num_images):
        display(Markdown(f"---\n### Image {i+1} Evaluation"))

        results = evaluate_pipeline(img_path_array[i], final_paths[i], img_index=i+1)
        evaluation_results.append(results)

        # Show before/after with metrics
        show_before_after_with_metrics(results, img_index=i+1, save_path="data/")

        # Print metrics
        print(f"\nImage {i+1} Metrics:")
        print("-" * 50)
        for metric in results['before']:
            before = results['before'][metric]
            after = results['after'][metric]
            imp = results['improvements'][metric]
            status = "✓" if imp['improved'] else "✗"
            print(f"  {metric}:")
            print(f"    Before: {before:.4f}")
            print(f"    After:  {after:.4f}")
            print(f"    Change: {imp['pct']:+.1f}% {status}")

    # Create summary
    display(Markdown("---"))
    summary_df = create_evaluation_summary(evaluation_results, save_path="data/")
else:
    print("Image paths not found. Run previous tasks first.")

# ============================================================================
# TASK 5 QUESTIONS
# ============================================================================

display(Markdown("---"))
display(Markdown("## Task 5 Questions"))

display(Markdown("""
### 1. Do the metrics agree with perceived improvement? Explain any mismatch.

**Answer:**

| Metric | Agreement with Perception | Explanation |
|--------|---------------------------|-------------|
| **Sharpness (VoL)** | Usually agrees | Higher VoL correlates well with perceived sharpness. However, noise also increases VoL, so a noisy image may score high but look worse. |
| **Contrast (Std)** | Generally agrees | Higher luminance std means more dynamic range used. May not capture local contrast improvements from CLAHE. |
| **Shadow Fraction** | Strong agreement | Directly measures what we want: fewer dark pixels = more visible shadow detail. |

**Potential Mismatches:**

1. **Sharpness vs Noise Trade-off**:
   - Enhancement may increase VoL due to amplified noise, not true sharpness
   - Denoising reduces VoL but improves perceived quality
   - Metric may decrease after denoising even though image looks better

2. **Contrast Saturation**:
   - Very high contrast (std) can look unnatural or "HDR-like"
   - Moderate contrast often looks more pleasing than maximum contrast
   - Metric doesn't distinguish "good" from "excessive" contrast

3. **Local vs Global**:
   - CLAHE improves local contrast but may not change global std much
   - Perceived improvement may not match metric change

---

### 2. Which metric is most informative for night enhancement?

**Answer:**

**Shadow Fraction is the most informative metric for night enhancement.**

**Reasons:**

1. **Directly addresses the core problem**: Night images suffer from underexposure with details hidden in shadows. Shadow fraction directly measures how much of the image is "too dark to see."

2. **Clear improvement direction**: Unlike sharpness (which can increase from noise) or contrast (which can be excessive), lower shadow fraction is always better for night enhancement.

3. **Interpretable**: "Reduced shadow pixels from 45% to 8%" is immediately meaningful, compared to "VoL increased from 150 to 280."

4. **Less affected by artifacts**: Sharpness can increase from noise or halos. Shadow fraction only improves when dark regions genuinely become visible.

**Metric Ranking for Night Enhancement:**

| Rank | Metric | Why |
|------|--------|-----|
| 1 | **Shadow Fraction** | Directly measures shadow recovery (primary goal) |
| 2 | **Contrast (Std)** | Indicates dynamic range expansion |
| 3 | **Sharpness (VoL)** | Useful but confounded by noise amplification |

**Best Practice:** Use shadow fraction as primary metric, but verify with visual inspection that increased contrast/sharpness don't introduce artifacts.
"""))